# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs by referencing their `@id` values.

In [ ]:
# List all record sets and their @id
print("Available Record Sets (by @id):\n")
record_sets = list(dataset.record_sets)
record_set_ids = []
for rs in record_sets:
    print(f"- Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dtype: {field.data_type})")
    print()
    record_set_ids.append(rs.id)
# For the rest of the notebook, we'll select the first (main) record set for examples
main_record_set_id = record_set_ids[0] if record_set_ids else None
if not main_record_set_id:
    raise ValueError('No record sets found in the dataset.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use `@id` values for reference.

In [ ]:
# Extract data from all record sets into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print(f"Columns for main record set (@id={main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Select a main numeric field for analysis
# Inspect available fields/columns
main_df = dataframes[main_record_set_id]
print("Sample data for column inspection:")
print(main_df.head())

# Try to select a likely numeric field using known quantitative descriptions from Croissant
# Let's choose e.g. "Coverage (%)" or "MW [Da]", common in proteomics

# Attempt auto-detection
possible_numeric_fields = [col for col in main_df.columns if main_df[col].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(main_df[col])]
if possible_numeric_fields:
    # Just pick first one for demonstration
    numeric_field = possible_numeric_fields[0]
    print(f"Numeric field selected: {numeric_field}\n")
else:
    raise ValueError('No numeric field found. Please update the field name for your dataset.')

# Filter data (e.g., threshold 10)
threshold = 10
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to select a likely group/categorical field (not numeric)
group_candidates = [col for col in main_df.columns if not pd.api.types.is_numeric_dtype(main_df[col]) and main_df[col].nunique() < main_df.shape[0] // 3 and main_df[col].nunique()>1]
if group_candidates:
    group_field = group_candidates[0]
    print(f"\nGrouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (mean values):")
    print(grouped_df.head())
else:
    print("No suitable group/categorical field detected for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field], bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# If grouping field selected, show boxplot
if 'group_field' in locals():
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and interpreted metadata from the FAIR\u2072 dataset using its Croissant schema.
- Explored available record sets, fields, and their associated `@id`s for semantic referencing.
- Loaded records into pandas DataFrames for analysis.
- Performed basic data filtering, normalization, and grouping using relevant fields by their `@id`s.
- Visualized the distribution and relationships of key quantitative variables.

This provides a template for deeper, domain-driven examination of mass spectrometry-based proteomics datasets using the Croissant standard and the `mlcroissant` library.
